In [8]:
"""
Setup dataframe for simulated priming study using RAW-C.

For each sentence pair (in each order), we create a "trial" where:
  - The prime is the first sentence
  - The target is the second sentence
  - The full context is: "{prime} {target}"
  - We'll later compute surprisal of the target given the prime

This gives us 2x the rows of the original dataset (one per order).
"""

import pandas as pd

# Load RAW-C
df = pd.read_csv("data/human/raw-c.csv")
df.head(3)

,word,sentence1,sentence2,same,ambiguity_type,disambiguating_word1,disambiguating_word2,version,Class,mean_relatedness,median_relatedness,diff,count,sd_relatedness,distance_bert,distance_elmo,se_relatedness,v1,v2,string
0,act,It was a desperate act.,It was a magic act.,False,Polysemy,desperate,magic,M1_a_M2_a,N,2.181818,2.0,0.181818,11,1.328020,0.204110,0.034093,0.400413,M1_a,M2_a,act
1,act,It was a desperate act.,It was a comedic act.,False,Polysemy,desperate,comedic,M1_a_M2_b,N,2.000000,2.0,0.000000,7,1.290994,0.215616,0.045927,0.487950,M1_a,M2_b,act
2,act,It was a humane act.,It was a magic act.,False,Polysemy,humane,magic,M1_b_M2_a,N,2.818182,3.0,0.181818,11,0.981650,0.191488,0.042351,0.295979,M1_b,M2_a,act


In [14]:
print(f"Original dataset: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Example sentence1: {df['sentence1'].iloc[0]}")
print(f"Example sentence2: {df['sentence2'].iloc[0]}")
print()

# Build order A: sentence1 is prime, sentence2 is target
order_a = df.copy()
order_a["prime"] = order_a["sentence1"]
order_a["target"] = order_a["sentence2"]
order_a["order"] = "s1_prime"

# Build order B: sentence2 is prime, sentence1 is target
order_b = df.copy()
order_b["prime"] = order_b["sentence2"]
order_b["target"] = order_b["sentence1"]
order_b["order"] = "s2_prime"

# Combine
priming_df = pd.concat([order_a, order_b], ignore_index=True)

# Create the full context string (prime + space + target)
priming_df["context"] = priming_df["prime"] + " " + priming_df["target"]

# Version with order: {prime_version}_{target_version}
# For s1_prime: prime=v1, target=v2 → v1_v2
# For s2_prime: prime=v2, target=v1 → v2_v1
priming_df["version_with_order"] = priming_df.apply(
    lambda r: f"{r['v1']}_{r['v2']}" if r["order"] == "s1_prime" else f"{r['v2']}_{r['v1']}",
    axis=1,
)

# Keep useful columns upfront for clarity
key_cols = [
    "word", "same", "ambiguity_type", "string", "version", "Class",
    "order", "version_with_order", "prime", "target", "context",
    "sentence1", "sentence2",
    "disambiguating_word1", "disambiguating_word2",
    "v1", "v2",
]
# Add any remaining columns not already in key_cols
remaining = [c for c in priming_df.columns if c not in key_cols]
priming_df = priming_df[key_cols + remaining]

print(f"Priming dataframe: {len(priming_df)} rows ({len(df)} pairs × 2 orders)")
print()

# Show a few examples
print("=== Example trials ===")
for i, row in priming_df.head(4).iterrows():
    print(f"\nRow {i} | word='{row['word']}' | order={row['order']} | same={row['same']}")
    print(f"  Prime:   {row['prime']}")
    print(f"  Target:  {row['target']}")
    print(f"  Context: {row['context']}")

# Verify: for each original row, we should have exactly 2 priming rows
print(f"\n=== Sanity checks ===")
print(f"Orders per original pair: {priming_df.groupby(['word', 'version']).size().unique()}")
print(f"Unique words: {priming_df['word'].nunique()}")
print(f"Unique versions: {priming_df['version'].nunique()}")


Original dataset: 672 rows
Columns: ['word', 'sentence1', 'sentence2', 'same', 'ambiguity_type', 'disambiguating_word1', 'disambiguating_word2', 'version', 'Class', 'mean_relatedness', 'median_relatedness', 'diff', 'count', 'sd_relatedness', 'distance_bert', 'distance_elmo', 'se_relatedness', 'v1', 'v2', 'string']
Example sentence1: It was a desperate act.
Example sentence2: It was a magic act.

Priming dataframe: 1344 rows (672 pairs × 2 orders)

=== Example trials ===

Row 0 | word='act' | order=s1_prime | same=False
  Prime:   It was a desperate act.
  Target:  It was a magic act.
  Context: It was a desperate act. It was a magic act.

Row 1 | word='act' | order=s1_prime | same=False
  Prime:   It was a desperate act.
  Target:  It was a comedic act.
  Context: It was a desperate act. It was a comedic act.

Row 2 | word='act' | order=s1_prime | same=False
  Prime:   It was a humane act.
  Target:  It was a magic act.
  Context: It was a humane act. It was a magic act.

Row 3 | word=

In [17]:
priming_df = priming_df[['word', 'same', 'ambiguity_type', 'string', 'version', 'Class', 'order',
       'version_with_order', 'prime', 'target', 'context', 'sentence1',
       'sentence2', 'disambiguating_word1', 'disambiguating_word2']]

In [20]:
priming_df.to_csv("data/raw/rawc_stimuli_with_order.csv", index=False)